In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from lifelines import AalenJohansenFitter, KaplanMeierFitter
from matplotlib.ticker import NullLocator, PercentFormatter

# 0. Configuration
AIDEV_REVISION = "68ed5f4"
DATASET_CUTOFF = pd.Timestamp("2025-08-01 23:59:59", tz="UTC")
ADMIN_WINDOW_DAYS = 60
PLOT_WINDOW_DAYS = 30
INSTANT_MERGE_CUTOFF_DAYS = 1 / 1440
TIMEPOINTS = [1, 3, 7, 30, 60]

FIGURES_DIR = Path("figures")
RESULTS_DIR = Path("results")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

target_agents = ["Human", "OpenAI_Codex", "Claude_Code", "Copilot", "Cursor", "Devin"]
agent_labels = {
    "Human": "Human",
    "OpenAI_Codex": "OpenAI Codex",
    "Claude_Code": "Claude Code",
    "Copilot": "Copilot",
    "Cursor": "Cursor",
    "Devin": "Devin",
}
color_map = {
    "Human": "#222222",
    "OpenAI_Codex": "#D1495B",
    "Claude_Code": "#D89000",
    "Copilot": "#2878B5",
    "Cursor": "#21866B",
    "Devin": "#8A60A8",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "0.65",
    "legend.fontsize": 9,
    "figure.titlesize": 16,
    "figure.titleweight": "bold",
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

followup_note = (
    f"Follow-up: up to {ADMIN_WINDOW_DAYS} days per PR, "
    f"ending no later than {DATASET_CUTOFF:%Y-%m-%d %H:%M:%S} UTC."
)


In [ ]:
# 1. Load data and select a common repository pool
df_ai_raw = load_dataset(
    "hao-li/AIDev", "pull_request", split="train", revision=AIDEV_REVISION,
).to_pandas()
df_human_raw = load_dataset(
    "hao-li/AIDev", "human_pull_request", split="train", revision=AIDEV_REVISION,
).to_pandas()

common_repos = set(df_ai_raw["repo_url"].dropna()) & set(
    df_human_raw["repo_url"].dropna()
)
df_valid = pd.concat(
    [df_ai_raw, df_human_raw.assign(agent="Human")], ignore_index=True,
)
df_valid = df_valid[
    df_valid["repo_url"].isin(common_repos)
    & df_valid["agent"].isin(target_agents)
].copy()

total_before = len(df_valid)
# Agent rows come first, so duplicate PRs retain their agent labels.
df_valid = df_valid.drop_duplicates(
    subset="html_url", keep="first",
).reset_index(drop=True)

print(f"Removed {total_before - len(df_valid):,} duplicate PRs.")
print(f"Retained {len(df_valid):,} PRs before checking creation times.")


In [ ]:
# 2. Define competing-risk events with administrative censoring
time_cols = ["created_at", "merged_at", "closed_at"]
for col in time_cols:
    df_valid[col] = pd.to_datetime(df_valid[col], errors="coerce", utc=True)

df_valid = df_valid[
    df_valid["created_at"].notna()
    & (df_valid["created_at"] <= DATASET_CUTOFF)
].copy()
df_valid["admin_end"] = (
    df_valid["created_at"] + pd.Timedelta(days=ADMIN_WINDOW_DAYS)
).clip(upper=DATASET_CUTOFF)

mask_merge = (
    df_valid["merged_at"].notna()
    & (df_valid["merged_at"] <= df_valid["admin_end"])
)
mask_close = (
    df_valid["merged_at"].isna()
    & df_valid["closed_at"].notna()
    & (df_valid["closed_at"] <= df_valid["admin_end"])
)

# Unmerged closure is treated as terminal; reopening histories are not reconstructed.
# 0 = censored, 1 = merged, 2 = closed without merging
df_valid["status_code"] = 0
df_valid.loc[mask_merge, "status_code"] = 1
df_valid.loc[mask_close, "status_code"] = 2

df_valid["obs_end"] = df_valid["admin_end"]
df_valid.loc[mask_merge, "obs_end"] = df_valid.loc[mask_merge, "merged_at"]
df_valid.loc[mask_close, "obs_end"] = df_valid.loc[mask_close, "closed_at"]
df_valid["lag_days"] = (
    df_valid["obs_end"] - df_valid["created_at"]
).dt.total_seconds() / 86400

# Validate observation times
assert df_valid["lag_days"].between(0, ADMIN_WINDOW_DAYS).all(), (
    "Missing or out-of-range lag_days; check the source timestamps."
)


In [ ]:
# 3. Descriptive statistics and fixed-time CIF estimates
status_counts = (
    df_valid.groupby(["agent", "status_code"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "Censored", 1: "Merged", 2: "Closed-Unmerged"})
    .reindex(columns=["Censored", "Merged", "Closed-Unmerged"], fill_value=0)
)
status_counts.index.name = "agent"
status_counts.columns.name = None
status_counts["Total_PRs"] = status_counts.sum(axis=1)
status_counts["Crude_Merge_Rate"] = status_counts["Merged"] / status_counts["Total_PRs"]

print(status_counts.to_string(formatters={"Crude_Merge_Rate": "{:.1%}".format}))

# Fixed-time CIF estimates use the full follow-up, not just the plotting window.
cif_records = []
ajf_table = AalenJohansenFitter(calculate_variance=False, seed=42)

for agent in target_agents:
    agent_data = df_valid[df_valid["agent"] == agent]
    if agent_data.empty:
        continue
    ajf_table.fit(
        durations=agent_data["lag_days"].to_numpy(),
        event_observed=agent_data["status_code"].to_numpy(),
        event_of_interest=1,
    )
    cif = ajf_table.cumulative_density_.iloc[:, 0]
    row = {"Agent": agent}
    for t in TIMEPOINTS:
        row[f"{t}d CIF"] = cif[cif.index <= t].iloc[-1]
    cif_records.append(row)

cif_table = pd.DataFrame(cif_records).set_index("Agent")
cif_table.index.name = "agent"
print("\nCumulative incidence at fixed timepoints:")
print(cif_table.to_string(float_format="{:.1%}".format))


In [ ]:
# 4. Sensitivity cohort: exclude merges occurring in less than one minute
df_main = df_valid.copy()

mask_instant_merge = (
    (df_valid["status_code"] == 1)
    & (df_valid["lag_days"] < INSTANT_MERGE_CUTOFF_DAYS)
)
df_sensitivity = df_valid[~mask_instant_merge].copy()


In [ ]:
# 5. AJ cumulative incidence: main and sensitivity analyses
# Variance is unnecessary here because confidence intervals are not displayed.
aj_fitter = AalenJohansenFitter(calculate_variance=False, seed=42)


def style_axis(ax, ylabel):
    ax.set_xlabel("Days since PR creation")
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, PLOT_WINDOW_DAYS)
    ax.set_ylim(0, 1.0)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
    ax.grid(True, linestyle="--", alpha=0.25)


def style_log_axis(ax, ylabel):
    ax.set_xscale("log")
    ax.set_xlim(1 / 1440, ADMIN_WINDOW_DAYS)
    ax.set_xticks([1 / 1440, 10 / 1440, 1 / 24, 6 / 24, 1, 7, 30, 60])
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xticklabels(["1 min", "10 min", "1 h", "6 h", "1 d", "7 d", "30 d", "60 d"])
    ax.set_xlabel("Time since PR creation (log scale)")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.0)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
    ax.grid(True, which="major", linestyle="--", alpha=0.25)


def plot_cohort(cohort_df, ax, title, use_log=True):
    for agent in target_agents:
        agent_data = cohort_df[cohort_df["agent"] == agent]
        if agent_data.empty:
            continue

        aj_fitter.fit(
            durations=agent_data["lag_days"].to_numpy(),
            event_observed=agent_data["status_code"].to_numpy(),
            event_of_interest=1,
        )
        aj_fitter.plot_cumulative_density(
            ax=ax,
            label=f"{agent_labels[agent]} (n={len(agent_data):,})",
            color=color_map[agent],
            linewidth=2.8 if agent == "Human" else 2.0,
            ci_show=False,
        )

    ax.set_title(title, pad=12)
    if use_log:
        style_log_axis(ax, "Cumulative incidence of merge")
    else:
        style_axis(ax, "Cumulative incidence of merge")
    ax.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.18),
        ncol=3, frameon=False, columnspacing=1.2, handlelength=2.0,
    )


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7.5))
plot_cohort(df_main, ax1, "Main analysis\nAll eligible PRs", use_log=True)
plot_cohort(
    df_sensitivity, ax2,
    "Sensitivity analysis\nExcluding merges < 1 minute",
    use_log=True,
)

fig.suptitle("PR merge probability over time: Human and AI agents", y=0.98)
fig.text(
    0.5, 0.045, followup_note + " Display: 1 minute–60 days (log scale).",
    ha="center", fontsize=10, color="0.35",
)
fig.text(
    0.5, 0.015,
    "Aalen–Johansen estimates in a common repository pool. "
    "Unmerged closure is treated as a competing event; confidence intervals are omitted.",
    ha="center", fontsize=10, color="0.35",
)
fig.subplots_adjust(left=0.055, right=0.985, bottom=0.28, top=0.84, wspace=0.18)
fig.savefig(FIGURES_DIR / "Aalen_Johansen_CIF.png")

fig_lin, (ax3, ax4) = plt.subplots(1, 2, figsize=(20, 7.5))
plot_cohort(df_main, ax3, "Main analysis\nAll eligible PRs", use_log=False)
plot_cohort(
    df_sensitivity, ax4,
    "Sensitivity analysis\nExcluding merges < 1 minute",
    use_log=False,
)

fig_lin.suptitle("PR merge probability over time: Human and AI agents", y=0.98)
fig_lin.text(
    0.5, 0.045, followup_note + f" Display: 0–{PLOT_WINDOW_DAYS} days (linear scale).",
    ha="center", fontsize=10, color="0.35",
)
fig_lin.text(
    0.5, 0.015,
    "Aalen–Johansen estimates in a common repository pool. "
    "Unmerged closure is treated as a competing event; confidence intervals are omitted.",
    ha="center", fontsize=10, color="0.35",
)
fig_lin.subplots_adjust(left=0.055, right=0.985, bottom=0.28, top=0.84, wspace=0.18)
fig_lin.savefig(FIGURES_DIR / "Aalen_Johansen_CIF_Linear.png")
plt.show()


In [ ]:
# 5.1. Summary of 30-day AJ estimates
df_dot = cif_table[["30d CIF"]].sort_values("30d CIF", ascending=True)

fig, ax = plt.subplots(figsize=(10, 4.5))

y_positions = np.arange(len(df_dot))
agents = df_dot.index
values = df_dot["30d CIF"].to_numpy()

ax.hlines(y_positions, xmin=0, xmax=values, color="0.85", linewidth=2.0, zorder=1)

colors = [color_map[a] for a in agents]
ax.scatter(values, y_positions, color=colors, s=140, zorder=2, edgecolors="none")

for y, val in zip(y_positions, values):
    ax.text(val + 0.015, y, f"{val:.1%}", va="center", fontweight="bold", color="0.2")

human_val = cif_table.loc["Human", "30d CIF"]
ax.axvline(
    human_val, color="0.5", linestyle="--", alpha=0.6,
    linewidth=1.2, zorder=0,
)
ax.set_ylim(-0.5, len(df_dot) - 0.5)

ax.set_yticks(y_positions)
ax.set_yticklabels([agent_labels[a] for a in agents], fontweight="bold")
ax.set_xlim(0, 1.0)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel("30-day cumulative incidence of merge (CIF)", labelpad=8)
ax.set_title("30-day cumulative merge probability", pad=15)
ax.grid(True, axis="x", linestyle="--", alpha=0.25)

ax.spines["left"].set_visible(False)

fig.text(
    0.5, 0.015,
    "Main cohort; AJ point estimates. Dashed line: Human reference. Confidence intervals are omitted.",
    ha="center", fontsize=9, color="0.35",
)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(FIGURES_DIR / "CIF_30d_Dot_Plot.png")
plt.show()


In [ ]:
# 6. KM vs AJ: competing-event handling within each author group
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()

kmf_all = KaplanMeierFitter(alpha=0.05)
ajf_all = AalenJohansenFitter(calculate_variance=True, alpha=0.05, seed=42)

for ax, agent in zip(axes, target_agents):
    agent_data = df_main[df_main["agent"] == agent]
    if agent_data.empty:
        ax.set_title(f"{agent_labels[agent]} (no data)")
        ax.set_axis_off()
        continue

    kmf_all.fit(
        durations=agent_data["lag_days"].to_numpy(),
        event_observed=(agent_data["status_code"] == 1).to_numpy(),
    )
    kmf_all.plot_cumulative_density(
        ax=ax,
        label="1 − KM (unmerged closure censored)",
        color="0.45",
        linestyle="--",
        linewidth=2.0,
        ci_show=False,
    )

    ajf_all.fit(
        durations=agent_data["lag_days"].to_numpy(),
        event_observed=agent_data["status_code"].to_numpy(),
        event_of_interest=1,
    )
    ajf_all.plot_cumulative_density(
        ax=ax,
        label="AJ (unmerged closure as competing event)",
        color=color_map[agent],
        linewidth=2.8 if agent == "Human" else 2.0,
        ci_show=True,
        ci_alpha=0.15,
    )

    close_rate = (agent_data["status_code"] == 2).mean()
    ax.set_title(
        f"{agent_labels[agent]} (n={len(agent_data):,})\n"
        f"Closed without merging: {close_rate:.1%} of PRs",
        pad=12,
    )
    style_axis(ax, "Cumulative probability estimate")
    ax.legend(loc="lower right")

fig.suptitle("KM vs AJ: effect of competing-event handling", y=0.98)
fig.text(
    0.5, 0.045, followup_note + f" First {PLOT_WINDOW_DAYS} days shown.",
    ha="center", fontsize=10, color="0.35",
)
fig.text(
    0.5, 0.025,
    "Main cohort. Shading: 95% pointwise AJ confidence intervals, "
    "not adjusted for repository clustering.",
    ha="center", fontsize=10, color="0.35",
)
fig.text(
    0.5, 0.005,
    "Closure percentages are observed proportions within follow-up, not CIF estimates. "
    "1 − KM and AJ represent different probability estimands.",
    ha="center", fontsize=10, color="0.35",
)
fig.tight_layout(rect=(0, 0.085, 1, 0.94))
fig.savefig(FIGURES_DIR / "Methodology_Comparison_ALL_Agents_KM_vs_AJ.png")
plt.show()


In [ ]:
# 7. Export summary tables
status_counts.to_csv(RESULTS_DIR / "summary_table.csv")
cif_table.to_csv(RESULTS_DIR / "cif_table.csv")
print(f"Saved summary_table.csv and cif_table.csv to {RESULTS_DIR}/")
